In [1]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile
from PythonFiles.constants import Constants as C
from pathlib import Path

c:\Users\mmapa\Desktop\Eti\ASR\ASR_project\.venv\Lib\site-packages\torchaudio\functional\functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [2]:
from PythonFiles.parsers import parse_phonemes, wav_to_logmel, windows_and_labels
from PythonFiles.parsers import PhonemeWindowDataset
from PythonFiles.NeuralModel import CRNN
from PythonFiles.NeuralModel import train, evaluate_tm

In [4]:
DATA_DIR = '../data/501-1000'   # adjust to your dataset location

# standardize=True: po prostu standardyzacja log mel spektrogramów

#silences_same=True: 
#  - jeśli False, to 'sil' i 'sp' są różnymi klasami (odpowiednio: cisza i pauza)
#  - jeśli True, to 'sil' i 'sp' są traktowane jako ta sama klasa (cisza/pauza)



dataset = PhonemeWindowDataset(DATA_DIR, max_files=300, verbose=True, standardize=True, silences_same=True)
print('X shape:', dataset.X.shape, 'y shape:', dataset.y.shape)

found 10969 TextGrid files under ../data/501-1000
  processed 100/300 files, 34430 windows
  processed 200/300 files, 69665 windows
  processed 300/300 files, 114818 windows
total windows: 114818
          S: 1718
          Z: 916
          a: 9255
          b: 1388
          c: 2128
          d: 2206
         dZ: 8
         dz: 258
        dzj: 467
          e: 7762
        eo5: 687
          f: 712
          g: 1576
          h: 1305
          i: 5969
         i2: 3710
          j: 3165
          k: 3826
          l: 2171
          m: 2893
          n: 4967
         n~: 211
          o: 7018
        oc5: 1770
          p: 3353
          r: 3007
          s: 3125
         sj: 1412
        sil: 19402
         sp: 0
          t: 5260
         tS: 1521
        tsj: 1555
          u: 2939
          v: 3770
          w: 1383
          z: 1914
         zj: 91
        oov: 0
X shape: torch.Size([114818, 128, 8]) y shape: torch.Size([114818])


In [5]:
n_val = max(1, int(0.2 * len(dataset)))
n_tr  = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_tr, n_val],
                                  generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = CRNN().to(device)
print(model)
print('params:', sum(p.numel() for p in model.parameters()))

CRNN(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (11): Dropout(p=0.2, inplace=False)
    (12): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
  )
  (rnn): LSTM(204

In [6]:
optim = torch.optim.NAdam(model.parameters(), lr=1e-3)
crit  = nn.CrossEntropyLoss()
EPOCHS = 15

for ep in range(1, EPOCHS + 1):
    model.train()
    tot_loss = tot_correct = tot = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = crit(logits, y)
        optim.zero_grad(); loss.backward(); optim.step()
        tot_loss    += loss.item() * y.size(0)
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot         += y.size(0)
    tr_loss = tot_loss / tot; tr_acc = tot_correct / tot

    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            vc += (model(X).argmax(1) == y).sum().item(); vt += y.size(0)
    print(f'epoch {ep:2d}  train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  val_acc={vc/vt:.3f}')

epoch  1  train_loss=1.8326  train_acc=0.481  val_acc=0.528
epoch  2  train_loss=1.4946  train_acc=0.564  val_acc=0.570
epoch  3  train_loss=1.3659  train_acc=0.598  val_acc=0.593
epoch  4  train_loss=1.2556  train_acc=0.626  val_acc=0.606
epoch  5  train_loss=1.1499  train_acc=0.656  val_acc=0.617
epoch  6  train_loss=1.0339  train_acc=0.688  val_acc=0.636
epoch  7  train_loss=0.9210  train_acc=0.720  val_acc=0.640
epoch  8  train_loss=0.8018  train_acc=0.756  val_acc=0.654


KeyboardInterrupt: 

In [7]:
CHECKPOINT_PATH = '8epok_300files.pt'

# --- save ---
torch.save({
    'model_state_dict': model.state_dict(),
    'labels':           C.LABELS,
    'config': {
        'sample_rate': C.SAMPLE_RATE,
        'n_fft':       C.N_FFT,
        'hop_length':  C.HOP_LENGTH,
        'n_mels':      C.N_MELS,
        'win_frames':  C.WIN_FRAMES,
        'shift_frames': C.SHIFT_FRAMES,
        'hidden':      64,
    },
}, CHECKPOINT_PATH)
print(f'saved to {CHECKPOINT_PATH}')

saved to 8epok_300files.pt
